In [ ]:
!nvidia-smi

In [ ]:
!uv pip uninstall 'keras' 'matplotlib' 'scikit-learn' 'tensorflow'

In [ ]:
!uv pip install -U --no-index \
    --find-links /kaggle/input/notebooks/chankhavu/vllm-0-23-0-py312-cu130/wheels \
    vllm flashinfer-jit-cache flashinfer-python flashinfer-cubin

In [ ]:
import os
import sys
import subprocess

import asyncio
import gc
import re
import json
import math
import time
import threading
from typing import Optional
from uuid import uuid4
from collections import Counter, defaultdict
from concurrent.futures import as_completed, ThreadPoolExecutor
from pathlib import Path

import pandas as pd
import polars as pl

from openai import OpenAI

os.environ['TRANSFORMERS_NO_TF'] = '1'
os.environ['TRANSFORMERS_NO_FLAX'] = '1'
os.environ['CUDA_VISIBLE_DEVICES'] = '0'
os.environ['TRITON_PTXAS_PATH'] = '/usr/local/cuda/bin/ptxas'
os.environ['VLLM_USE_FLASHINFER_MOE_FP8'] = '1'

In [ ]:
!cp /kaggle/input/notebooks/chankhavu/ipython-sandbox/sandbox_client.py ./sandbox_client.py
!cp /kaggle/input/notebooks/chankhavu/ipython-sandbox/sandbox_server.py ./sandbox_server.py
from sandbox_client import LocalSandbox

In [ ]:
SANDBOX_PREIMPORT = (
    "import math\n"
    "import numpy\n"
    "import sympy\n"
    "import itertools\n"
    "import collections\n"
    "import mpmath\n"
    "mpmath.mp.dps = 64\n"
)

In [ ]:
sandbox_server_script = "sandbox_server.py"

print('Starting sandbox server...')
sandbox_log_file = open('sandbox_server.log', 'w')
sandbox_process = subprocess.Popen(
    [sys.executable, sandbox_server_script],
    stdout=sandbox_log_file,
    stderr=subprocess.STDOUT,
    start_new_session=True
)

# Wait for sandbox to be ready
_sandbox_check = LocalSandbox(host="localhost", port=6000)

for _attempt in range(30):
    rc = sandbox_process.poll()
    if rc is not None:
        sandbox_log_file.flush()
        with open('sandbox_server.log', 'r') as f:
            logs = f.read()
        raise RuntimeError(f'Sandbox server died with code {rc}. Logs:\n{logs}\n')
    if _sandbox_check._check_ready(timeout=2):
        break
    time.sleep(1)
else:
    raise RuntimeError('Sandbox server failed to start (timeout).\n')

print('Sandbox server is ready.\n')

In [ ]:
def _preload_model_weights(model_path):

    print(f'Loading model weights from {model_path} into OS Page Cache...')
    start_time = time.time()

    files_to_load = []
    total_size = 0

    for root, _, files in os.walk(model_path):
        for file_name in files:
            file_path = os.path.join(root, file_name)

            if os.path.isfile(file_path):
                files_to_load.append(file_path)
                total_size += os.path.getsize(file_path)

    def _read_file(path: str) -> None:

        with open(path, 'rb') as file_object:
            while file_object.read(1024 * 1024 * 1024):
                pass

    with ThreadPoolExecutor(max_workers=8) as executor:
        list(executor.map(_read_file, files_to_load))

    elapsed = time.time() - start_time
    print(f'Processed {len(files_to_load)} files ({total_size / 1e9:.2f} GB) in {elapsed:.2f} seconds.\n')

_preload_model_weights("/kaggle/input/models/chankhavu/smolmo-32b-sft-merged-proofpilot/transformers/nvfp4/2")

In [ ]:
import torch
print(torch.cuda.get_device_capability())  # should be (12, 0)

! ls /usr/local/ | grep cuda
! find /usr/local/lib/python3.12/dist-packages/nvidia/cu13 -name "nvcc" 2>/dev/null
! ls /usr/local/lib/python3.12/dist-packages/nvidia/cu13/
! ls /usr/local/lib/python3.12/dist-packages/nvidia/cu13/bin/ 2>/dev/null
! /usr/local/lib/python3.12/dist-packages/nvidia/cu13/bin/nvcc --list-gpu-arch 2>/dev/null

In [ ]:
cuda13 = "/usr/local/lib/python3.12/dist-packages/nvidia/cu13"
os.environ["CUDA_HOME"] = cuda13
os.environ["CUDA_PATH"] = cuda13
os.environ["PATH"] = f"{cuda13}/bin:" + os.environ.get("PATH", "")
os.environ["LD_LIBRARY_PATH"] = f"{cuda13}/lib:" + os.environ.get("LD_LIBRARY_PATH", "")
os.environ["TORCH_CUDA_ARCH_LIST"] = "12.0"  # SM 120 blackwell

In [ ]:
# rm -rf /root/.cache/vllm/torch_compile_cache/

In [ ]:
import os
os.environ["LD_LIBRARY_PATH"] = (
    "/usr/local/lib/python3.12/dist-packages/nvidia/cu13/lib:"
    + os.environ.get("LD_LIBRARY_PATH", "")
)

# VAST_AI config:
# --max-model-len 65536 --kv-cache-dtype fp8 --trust-remote-code --enable-prefix-caching --enable-chunked-prefill --async-scheduling
vllm_cmd = [
    "vllm", "serve", "/kaggle/input/models/chankhavu/smolmo-32b-sft-merged-proofpilot/transformers/nvfp4/2",
    '--served-model-name', "chankhavu/smolmo-32b-sft-merged-proofpilot-nvfp4",
    '--tensor-parallel-size', '1',
    # '--max-num-seqs', str(CFG.max_num_seqs),
    '--gpu-memory-utilization', "0.95",
    '--host', '0.0.0.0',
    '--port', "8000",
    '--max-model-len', "65536",
    '--kv-cache-dtype', 'fp8',
    '--trust-remote-code',
    '--enable-chunked-prefill',
    '--enable-prefix-caching',
    '--async-scheduling',
]
print("VLLM cmd:", vllm_cmd)

vllm_log_file = open('vllm_server.log', 'w')
vllm_process = subprocess.Popen(
    vllm_cmd,
    stdout=vllm_log_file,
    stderr=subprocess.STDOUT,
    start_new_session=True
)

openai_client = OpenAI(
    base_url=f'http://0.0.0.0:8000/v1',
    api_key='sk-local',
    timeout=960
)

print('Waiting for vLLM server...')
start_time = time.time()

for _ in range(240):
    return_code = vllm_process.poll()

    if return_code is not None:
        vllm_log_file.flush()

        with open('vllm_server.log', 'r') as f:
            logs = f.read()

        raise RuntimeError(f'Server died with code {return_code}. Full logs:\n{logs}\n')

    try:
        openai_client.models.list()
        elapsed = time.time() - start_time
        print(f'Server is ready (took {elapsed:.2f} seconds).\n')

        break

    except Exception:
        time.sleep(1)

else:
    raise RuntimeError('Server failed to start (timeout).\n')

In [ ]:
%%writefile kaggle_solver.py
#!/usr/bin/env python3
"""
Self-contained inference code for the proof solver — everything EXCEPT the code-execution sandbox
(server + client), consolidated into one file for easy transfer to a Kaggle notebook.

Public API:
    cfg = SolveConfig(server_addr="host:port", api_key="...")   # champ defaults baked in
    result = solve(problem_text, cfg)        # -> {"best": <str>, "all": <pandas.DataFrame>}

Three layers, top to bottom:
  1. ENGINE         — collect_rollout(): one streaming rollout, self-parsed <function_calls> tool loop,
                      deadline-based abort (stream close -> vLLM aborts the decode, frees the GPU).
  2. PROMPTS/CLIENT — byte-exact proof/analysis/refine prompts, build_messages, extract_result,
                      build_client (OpenAI-compatible client pointing at the vLLM server).
  3. PIPELINE       — SolveConfig + solve(): solution -> self-verify -> (optional) refine, ranked by
                      the model's own self-verification (no ground truth in the loop).

THE ONE EXTERNAL DEPENDENCY you must wire on Kaggle: `sandbox_client.LocalSandbox`, an async client to
your code-execution sandbox exposing:
    await execute_code(generated_code=str, session_id=str, timeout=int, max_output_characters=int)
        -> (output_dict_with_stdout_stderr, _)
    await delete_session(session_id)
    await close()
Replace the placeholder import below with your real client.

Best config (DEFAULT): 8 tool solutions, 3 self-verify each, 1 refine round (refine the top-4 eligible
solutions — eligible = neither unanimous-7 nor unanimous-0 — 2 refine rollouts each; early-exit if any
solution is already unanimous-7), solution cap 1800s, verify cap 360s, refine cap 1800s, max_parallel 16.
The CSV runner (main) overrides the per-phase caps with a greedy dynamic budget (floor 30, ceiling 90 min).

──────────────────────────────────────────────────────────────────────────────────────────────────
HOW TO RUN (CSV in -> CSV out) — best config is the DEFAULT, so you only pass paths + budget:

    python3 kaggle_solver.py \
        --csv        /path/to/INPUT.csv \
        --out_csv    /path/to/OUTPUT.csv \
        --outdir     /path/to/run_artifacts \
        --sandbox_addr 127.0.0.1:6001 \
        --endpoints  ENDPOINTS.yaml \
        --total_budget_s 21600 \
        --min_problem_s 1800 --max_problem_s 5400 --verify_reserve_s 720

  Or pin the endpoint explicitly instead of --endpoints:
        --server_addr HOST:PORT --api_key KEY --model_name MODEL

  - --total_budget_s is the WHOLE-RUN wall-clock (seconds), split greedily across all problems in the
    CSV and clamped per-problem to [min_problem_s, max_problem_s] = [30 min, 90 min]; fast problems bank
    surplus for later ones. SET IT — it defaults to 0 (per-phase caps only, no cross-problem reservation).
    Rule of thumb: ~3600 * num_problems.
  - The config flags (--n_tool_solutions 8 --n_tool_verify 3 --refine_rounds 1 --n_refine_candidates 4
    --n_refine_tool 2) are the DEFAULTS; pass them only to override.

INPUT  CSV: needs an id column (id|Problem ID|problem_id|ID) and a problem column
            (problem|Problem|question|Question|statement). One problem per row.
OUTPUT CSV: one row per input problem (every line — incl. failures/dupes), rewritten ATOMICALLY after
            each problem (a kill mid-run leaves a valid partial CSV). Columns:
            id, solution, best_self_verify, best_chars, elapsed_s, error
            `solution` is the full best proof text (== <outdir>/<id>/best.txt); `error` blank on success.

Programmatic use instead of the CSV runner (e.g. a Kaggle predict() hook):
    import kaggle_solver as K, time
    cfg = K.SolveConfig(server_addr="HOST:PORT", api_key="...", model_name="...",
                        sandbox_host="127.0.0.1", sandbox_port=6001)
    def predict(problem: str) -> str:
        return K.solve(problem, cfg, problem_deadline=time.time() + 3600)["best"]

PRE-FLIGHT: vLLM endpoint + sandbox (port 6001) both up, sandbox_client.py on the path, ONE solver at a
time (concurrent runs starve the GPU). Requires pip: openai, httpx, pandas.
──────────────────────────────────────────────────────────────────────────────────────────────────
"""

from __future__ import annotations

import argparse
import ast
import asyncio
import csv
import json
import logging
import os
import re
import statistics
import time
from collections import defaultdict
from concurrent.futures import ThreadPoolExecutor, as_completed
from concurrent.futures import TimeoutError as FutureTimeoutError
from dataclasses import dataclass
from pathlib import Path
from typing import Any, Callable, Optional
from uuid import uuid4

import httpx
import pandas as pd
from openai import OpenAI

# --- THE ONE EXTERNAL DEPENDENCY: the code-execution sandbox client ------------------------------
# Direct import — NO placeholder. We fail honestly at import time if the sandbox client is missing
# (a silent placeholder would let the whole batch "succeed" with empty submissions). On Kaggle, make
# `sandbox_client` importable (e.g. copy it onto the path). RUNTIME sandbox errors are caught inside
# the rollout loop and surfaced as "[ERROR] ..." tool output; only the IMPORT is allowed to be fatal.
from sandbox_client import LocalSandbox  # noqa: F401


log = logging.getLogger("kaggle_solver")

DEFAULT_MODEL = "chankhavu/smolmo-32b-sft-merged-proofpilot-nvfp4"

# ==================================================================================================
# LAYER 1 — ROLLOUT ENGINE
# ==================================================================================================
# We deliberately do NOT rely on vLLM's --tool-call-parser olmo3 (it corrupts multi-line code= args).
# The model emits its trained format as plain text:
#     <think>...</think><function_calls>stateful_python_code_exec(code="...")</function_calls>
# and we self-parse the call (ast, regex fallback) and execute it in the sandbox. tools= only controls
# whether the <functions> schema is rendered into the prompt; the conversation loop is identical with
# or without it, so a no-schema ("no-tool") prompt that still emits a call is honored.

FUNCTION_CALLS_OPEN = "<function_calls>"
FUNCTION_CALLS_CLOSE = "</function_calls>"
END_OF_TEXT = "<|endoftext|>"
END_OF_TURN = "<|im_end|>"

# Libraries pre-imported into every sandbox session so tool code can use them without boilerplate.
SANDBOX_PREIMPORT = (
    "import math\n"
    "import numpy\n"
    "import sympy\n"
    "import itertools\n"
    "import collections\n"
    "import mpmath\n"
    "mpmath.mp.dps = 64\n"
)


def extract_tool_calls(assistant_text: str) -> list[Optional[str]]:
    """Extract the ``code`` argument from every ``<function_calls>`` block in ``assistant_text``.

    One entry per block: the code string, or ``None`` if it could not be parsed. Robust to multi-line
    code and nested quotes (ast-parse the single call; regex fallback if that raises).
    """
    extracted: list[Optional[str]] = []
    block_pattern = re.escape(FUNCTION_CALLS_OPEN) + r"(.*?)" + re.escape(FUNCTION_CALLS_CLOSE)
    for block in re.finditer(block_pattern, assistant_text, re.DOTALL):
        call_source = block.group(1).strip()
        code = _parse_code_with_ast(call_source)
        if code is None:
            code = _parse_code_with_regex(call_source)
        extracted.append(code)
    return extracted


def _parse_code_with_ast(call_source: str) -> Optional[str]:
    """Parse ``stateful_python_code_exec(code=...)`` via the AST and return the ``code`` literal."""
    match = re.search(r"stateful_python_code_exec\s*\(.*\)", call_source, re.DOTALL)
    if not match:
        return None
    try:
        node = ast.parse(match.group(0).strip(), mode="eval").body
    except SyntaxError:
        return None
    if not isinstance(node, ast.Call):
        return None
    for keyword in node.keywords:
        if keyword.arg == "code":
            try:
                return ast.literal_eval(keyword.value)
            except (ValueError, SyntaxError):
                return None
    if node.args:
        try:
            return ast.literal_eval(node.args[0])
        except (ValueError, SyntaxError):
            return None
    return None


def _parse_code_with_regex(call_source: str) -> Optional[str]:
    """Last-resort extraction of a ``code='...'`` / ``code="..."`` argument when the AST parse fails."""
    match = re.search(r"code\s*=\s*(['\"])(.*?)\1\s*\)?\s*$", call_source, re.DOTALL)
    if not match:
        return None
    quote, body = match.group(1), match.group(2)
    try:
        return ast.literal_eval(quote + body + quote)
    except (ValueError, SyntaxError):
        # Best-effort: usually genuine raw code the AST choked on (real newlines inside quotes, which is
        # not a valid single-quoted literal but IS runnable). Return it; the sandbox surfaces a real
        # SyntaxError if it was truly broken. (~0.5% of calls reach this fallback.)
        return body


# Exception class line ("SyntaxError: ...") at line start. SyntaxError carries no "Traceback" line.
_ERROR_LINE = re.compile(r"(?m)^\s*\w*(?:Error|Exception)\s*:")


def _looks_like_error(tool_output: str) -> bool:
    """Heuristic for whether a sandbox result represents a failure."""
    return (
        tool_output.startswith("[ERROR]")
        or "Traceback (most recent call last)" in tool_output
        or "timed out" in tool_output
        or bool(_ERROR_LINE.search(tool_output))
    )


async def _execute_in_sandbox(sandbox, session_id, code, timeout, max_output_chars) -> str:
    """Run ``code`` as an IPython cell in the sandbox and return its combined stdout/stderr."""
    output, _ = await sandbox.execute_code(
        generated_code=code,
        session_id=session_id,
        timeout=timeout,
        max_output_characters=max_output_chars,
    )
    stdout, stderr = output.get("stdout", ""), output.get("stderr", "")
    if stderr:
        return f"{stdout.rstrip()}\n{stderr}" if stdout.strip() else stderr
    return stdout if stdout.strip() else "[WARN] No output. Use print() to see results."


async def collect_rollout(
    *,
    client,
    model: str,
    messages: list[dict],
    tools: Optional[list[dict]] = None,
    sandbox=None,
    temperature: float = 1.0,
    top_p: float = 0.95,
    max_tokens: int = 0,
    max_turns: int = 64,
    python_timeout: int = 60,
    max_output_chars: int = 3000,
    deadline: Optional[float] = None,
    extra_body: Optional[dict] = None,
) -> dict[str, Any]:
    """Run a single rollout and return a trace dict.

    ``tools`` controls ONLY whether the <functions> schema is rendered (None => not rendered); the loop
    is identical either way (we always self-parse + execute emitted calls). ``deadline`` (epoch secs)
    aborts generation when exceeded, between turns and mid-stream (closing the stream disconnects the
    HTTP request and vLLM aborts the in-flight decode). Returns ``finish_reason`` + ``generation``
    always; ``finish_reason == "error"`` (with ``error``) instead of raising, so callers can continue.
    """
    render_functions = tools is not None
    started_at = time.time()
    conversation = [dict(message) for message in messages]
    request_extra_body = {"skip_special_tokens": False}
    if extra_body:
        request_extra_body.update(extra_body)

    sandbox_session_id = uuid4().hex if sandbox is not None else None
    finish_reason = "max_turns"
    total_completion_tokens = 0
    last_prompt_tokens = 0
    num_tool_calls = 0
    num_tool_errors = 0
    turn_log: list[dict] = []

    try:
        if sandbox_session_id is not None:
            try:
                await sandbox.execute_code(
                    generated_code=SANDBOX_PREIMPORT, session_id=sandbox_session_id, timeout=30
                )
            except Exception:
                pass  # pre-import is best-effort; the model's own imports usually suffice.

        for turn_index in range(max_turns):
            if deadline is not None and time.time() > deadline:
                finish_reason = "deadline"
                break

            request = {
                "model": model,
                "messages": conversation,
                "temperature": temperature,
                "top_p": top_p,
                # Stream so a deadline can abort the decode (closing the stream -> vLLM aborts, frees GPU).
                "stream": True,
                "stream_options": {"include_usage": True},
                "extra_body": request_extra_body,
            }
            if render_functions:
                # tools= makes the template render the schema; tool_choice="none" avoids needing the
                # server-side parser (we parse the emitted calls ourselves).
                request["tools"] = tools
                request["tool_choice"] = "none"
            if max_tokens and max_tokens > 0:
                request["max_completion_tokens"] = max_tokens

            assistant_text = ""
            reasoning_text = ""
            turn_finish_reason = None
            aborted = False
            stream = client.chat.completions.create(**request)
            try:
                for chunk in stream:
                    if deadline is not None and time.time() > deadline:
                        aborted = True
                        break
                    usage = getattr(chunk, "usage", None)
                    if usage is not None:
                        total_completion_tokens += usage.completion_tokens or 0
                        last_prompt_tokens = usage.prompt_tokens or last_prompt_tokens
                    if not chunk.choices:
                        continue  # usage-only final chunk carries no choices
                    choice = chunk.choices[0]
                    delta = choice.delta
                    if delta is not None:
                        # Most configs keep the whole <think>...</think> in `content`; a server with a
                        # reasoning parser splits it into `reasoning_content` — capture both.
                        if getattr(delta, "reasoning_content", None):
                            reasoning_text += delta.reasoning_content
                        if delta.content:
                            assistant_text += delta.content
                    if choice.finish_reason:
                        turn_finish_reason = choice.finish_reason
            finally:
                try:
                    stream.close()  # disconnect -> server-side abort if still generating
                except Exception:
                    pass
                # Commit whatever was decoded THIS turn into `conversation` *inside* the finally, BEFORE
                # any exception (read-timeout stall-net, mid-stream server death) can propagate to the
                # outer except. Otherwise the in-flight turn's reasoning+content is silently discarded
                # and `generation` rebuilds to "" — defeating the raw-generation fallback. (If the server
                # split reasoning out, splice it back inline so </think>-based extraction still works.)
                if reasoning_text and "</think>" not in assistant_text:
                    assistant_text = f"<think>{reasoning_text}</think>" + assistant_text
                if assistant_text:
                    conversation.append({"role": "assistant", "content": assistant_text})

            if aborted:
                turn_log.append({"turn": turn_index, "tool_calls": 0})
                finish_reason = "deadline"
                break

            model_made_a_call = FUNCTION_CALLS_OPEN in assistant_text
            if not model_made_a_call:
                turn_log.append({"turn": turn_index, "tool_calls": 0})
                if turn_finish_reason == "length":
                    finish_reason = "length"
                elif turn_finish_reason is None:
                    finish_reason = "incomplete"
                else:
                    finish_reason = "stop"
                break

            calls = extract_tool_calls(assistant_text)
            if not calls:
                # `<function_calls>` opened but no parseable closed call — i.e. the turn was cut off
                # mid-call (finish_reason="length") with an unterminated block. Don't silently spin to
                # the next turn re-feeding the broken half-call (it can loop to max_turns). End here.
                turn_log.append({"turn": turn_index, "tool_calls": 0})
                finish_reason = "length" if turn_finish_reason == "length" else "stop"
                break
            for call_index, code in enumerate(calls):
                num_tool_calls += 1
                if sandbox_session_id is None:
                    result = "[ERROR] No code-execution sandbox is available."
                elif not code:
                    result = "[ERROR] Could not parse code from the function call."
                else:
                    try:
                        # Hard wall-clock cap so a wedged sandbox (dead server / hung HTTP) can't block
                        # the rollout past its deadline — the sandbox's own `timeout` is only advisory.
                        result = await asyncio.wait_for(
                            _execute_in_sandbox(
                                sandbox, sandbox_session_id, code, python_timeout, max_output_chars
                            ),
                            timeout=python_timeout + 30.0,
                        )
                    except asyncio.TimeoutError:
                        result = f"[ERROR] sandbox call exceeded {python_timeout + 30.0:.0f}s wall-clock"
                    except Exception as exc:
                        result = f"[ERROR] {type(exc).__name__}: {exc}"
                if _looks_like_error(result):
                    num_tool_errors += 1
                conversation.append(
                    {
                        "role": "tool",
                        "tool_call_id": f"call_{turn_index}_{call_index}",
                        "name": "stateful_python_code_exec",
                        "content": result,
                    }
                )
            turn_log.append({"turn": turn_index, "tool_calls": len(calls)})
        else:
            finish_reason = "max_turns"

    except Exception as exc:
        return {
            "finish_reason": "error",
            "error": f"{type(exc).__name__}: {exc}",
            "generation": "".join(
                m["content"] for m in conversation if m.get("role") == "assistant" and m.get("content")
            ),
            "conversation": conversation,
            "num_turns": len(turn_log),
        }
    finally:
        if sandbox_session_id is not None:
            try:
                await sandbox.delete_session(sandbox_session_id)
            except Exception:
                pass
            try:
                await sandbox.close()
            except Exception:
                pass

    generation = "".join(
        m["content"] for m in conversation if m.get("role") == "assistant" and m.get("content")
    )
    return {
        "finish_reason": finish_reason,
        "truncated": finish_reason in ("length", "max_turns"),
        "num_turns": len(turn_log),
        "num_tool_calls": num_tool_calls,
        "num_tool_errors": num_tool_errors,
        "num_completion_tokens": total_completion_tokens,
        "num_prompt_tokens": last_prompt_tokens,
        "generation": generation,
        "conversation": conversation,
        "turns": turn_log,
        "generation_time": round(time.time() - started_at, 2),
    }


# ==================================================================================================
# LAYER 2 — PROMPTS, CLIENT, ANSWER EXTRACTION
# ==================================================================================================
# Byte-exact prompt literals (the model is extremely prompt-sensitive). Modes: proof / analysis (grade
# 0/1/6/7) / boxed / refine, each in no-tool and tool variants. Tool variants pass FUNCTIONS_SCHEMA so
# the baked chat template renders the <functions> block.

SYSTEM_NO_TOOL = (
    "You are an expert mathematical assistant. Provide rigorous, complete proofs. "
    "You are not allowed to use tools."
)
SYSTEM_BOXED_NO_TOOL = SYSTEM_NO_TOOL

SYSTEM_WITH_TOOL = (
    "You are an expert mathematical assistant. Provide rigorous, complete solutions. "
    "You are provided with function signatures within <functions></functions> XML tags. "
    "You may call one or more functions to assist with the user query. Output any function calls "
    "within <function_calls></function_calls> XML tags. Don't make assumptions about what values to "
    "plug into functions."
)

FUNCTIONS_SCHEMA_JSON = (
    '[{"type": "function", "function": {"name": "stateful_python_code_exec", "description": "Call '
    'this function to execute Python code in a stateful Jupyter notebook environment. Python will '
    'respond with the output of the execution or time out after 120.0 seconds.", "parameters": '
    '{"type": "object", "properties": {"code": {"type": "string", "description": "Code to execute"}}, '
    '"required": ["code"]}}}]'
)
FUNCTIONS_SCHEMA = json.loads(FUNCTIONS_SCHEMA_JSON)

PROOF_USER_PREFIX = (
    "Your task is to solve a given problem. The problem may ask you to prove a statement, or ask for "
    "an answer. If finding an answer is required, you should come up with the answer, and your final "
    "solution should also be a rigorous proof of that answer being valid.\n\nYour final solution to "
    "the problem should be exceptionally comprehensive and easy-to-follow, which will be rated "
    "according to the following evaluation instruction:\n\n```txt\nHere is the instruction to "
    "evaluate the quality of a solution to a problem. The problem may ask for a proof of statement, "
    "or ask for an answer. If finding an answer is required, the solution should present the answer, "
    "and it should also be a rigorous proof of that answer being valid.\n\nPlease evaluate the "
    "solution and score it according to the following criteria:\n- If the solution is completely "
    "correct, with all steps executed properly and clearly demonstrated, then the score is 1\n- If "
    "the solution is generally correct, but with some details omitted or minor errors, then the "
    "score is 0.5\n- If the solution does not actually address the required problem, contains fatal "
    "errors, or has severe omissions, then the score is 0\n\nAdditionally, referencing anything from "
    "any paper does not save the need to prove the reference. It's okay IF AND ONLY IF the solution "
    "also presents a valid proof of the reference argument(s); otherwise, if the solution omits the "
    "proof or if the proof provided is not completely correct, the solution should be scored "
    "according to the criteria above, and definitely not with a score of 1\n```\n\nYour final "
    "response should be in the following format:\n\n## Solution // Your final solution should start "
    "with this exact same markdown title\n... // Your final solution to the problem here. You should "
    "try your best to optimize the quality of your solution according to the evaluation instruction "
    "above before finalizing it here.\n\n---\n\nHere is your task input:\n\n## Problem\n"
)

ANALYSIS_USER_PREFIX = (
    "Carefully analyze the given problem statement and the proposed solution, and then write out "
    "your analysis regarding the correctness of the proposed solution. \n\nAfter the analysis, you "
    "must provide a score based on the following grading scale:\n\n- 0: Incorrect - The solution is "
    "completely incorrect or irrelevant.\n- 1: Partial - The solution is partially correct but has "
    "significant errors or omissions.\n- 6: Almost - The solution is almost correct but contains "
    "minor errors or inaccuracies.\n- 7: Correct - The solution is fully correct and complete.\n\n\n"
    "Problem:\n"
)
ANALYSIS_SOLUTION_SEPARATOR = "\n\nSolution:\n"
ANALYSIS_USER_SUFFIX = (
    "Analyze the solution carefully, then provide your grade as a single number (0, 1, 6, or 7) in "
    "\\boxed{{}}."
)

# DOUBLE-brace in the no-tool boxed prompt, SINGLE-brace in the tool boxed prompt — intentional.
BOXED_NO_TOOL_PREFIX = "Please reason step by step, and put your final answer within \\boxed{{}}.\n\n"
BOXED_WITH_TOOL_PREFIX = "Please reason step by step, and put your final answer within \\boxed{}.\n\n"

# Refine task: the refine USER turn is the proof prompt VERBATIM (PROOF_USER_PREFIX + problem) with two
# extra input sections and a closing instruction appended.
REFINE_PREV_SOLUTION_SEP = "\n\n## Previous Solution\n"
REFINE_EVALUATION_SEP = "\n\n## Evaluation\n"
REFINE_SUFFIX = (
    "\n\nThe ## Previous Solution above is an earlier attempt at this problem, and ## Evaluation is "
    "an assessment of that attempt's quality. Produce an improved solution: address every error, gap, "
    "and omission identified in the evaluation, and otherwise strengthen the rigor, completeness, and "
    "clarity of the argument. Present your improved final solution using the ## Solution format "
    "described above."
)


def build_proof_user(problem: str) -> str:
    return PROOF_USER_PREFIX + problem


def build_refine_user(problem: str, solution: str, evaluation: str) -> str:
    return (PROOF_USER_PREFIX + problem + REFINE_PREV_SOLUTION_SEP + solution
            + REFINE_EVALUATION_SEP + evaluation + REFINE_SUFFIX)


def build_analysis_user(problem: str, solution: str) -> str:
    return ANALYSIS_USER_PREFIX + problem + ANALYSIS_SOLUTION_SEPARATOR + solution + "\n\n" + ANALYSIS_USER_SUFFIX


def build_boxed_no_tool_user(problem: str) -> str:
    return BOXED_NO_TOOL_PREFIX + problem


def build_boxed_with_tool_user(problem: str) -> str:
    return BOXED_WITH_TOOL_PREFIX + problem


def _chat(system_content: str, user_content: str) -> list[dict]:
    return [
        {"role": "system", "content": system_content},
        {"role": "user", "content": user_content},
    ]


def build_messages(
    task: str, problem: str, solution: str = "", evaluation: str = ""
) -> tuple[list[dict], Optional[list[dict]]]:
    """Return ``(messages, tools)`` for ``task``. ``tools`` is ``None`` for no-tool modes and the parsed
    FUNCTIONS schema for tool modes. ``solution`` required by analysis/refine; ``evaluation`` by refine."""
    if task == "proof_notool":
        return _chat(SYSTEM_NO_TOOL, build_proof_user(problem)), None
    if task == "proof_tool":
        return _chat(SYSTEM_WITH_TOOL, build_proof_user(problem)), FUNCTIONS_SCHEMA
    if task == "analysis_notool":
        return _chat(SYSTEM_NO_TOOL, build_analysis_user(problem, solution)), None
    if task == "analysis_tool":
        return _chat(SYSTEM_WITH_TOOL, build_analysis_user(problem, solution)), FUNCTIONS_SCHEMA
    if task == "boxed_notool":
        return _chat(SYSTEM_BOXED_NO_TOOL, build_boxed_no_tool_user(problem)), None
    if task == "boxed_tool":
        return _chat(SYSTEM_WITH_TOOL, build_boxed_with_tool_user(problem)), FUNCTIONS_SCHEMA
    if task == "refine_notool":
        return _chat(SYSTEM_NO_TOOL, build_refine_user(problem, solution, evaluation)), None
    if task == "refine_tool":
        return _chat(SYSTEM_WITH_TOOL, build_refine_user(problem, solution, evaluation)), FUNCTIONS_SCHEMA
    raise ValueError(f"unknown task: {task!r}")


# --- answer extraction ----------------------------------------------------------------------------

_VALID_ANALYSIS_SCORES = {0, 1, 6, 7}
_THINK_CLOSE = "</think>"


def _iter_boxed(text: str):
    """Yield the brace-BALANCED contents of each ``\\boxed{...}`` (handles nesting). Unbalanced skipped."""
    needle = "\\boxed{"
    i = 0
    while True:
        j = text.find(needle, i)
        if j < 0:
            return
        k = j + len(needle)
        depth, start = 1, k
        while k < len(text) and depth:
            depth += {"{": 1, "}": -1}.get(text[k], 0)
            k += 1
        if depth == 0:
            yield text[start:k - 1]
        i = k


def _last_boxed(text: str) -> Optional[str]:
    boxes = [b.strip() for b in _iter_boxed(text)]
    return boxes[-1] if boxes else None


def extract_result(task: str, generation: str) -> dict[str, Any]:
    """Pull the task-specific field out of the full generation text."""
    if task.startswith("proof") or task.startswith("refine"):
        return {"solution": _extract_proof_solution(generation)}
    if task.startswith("analysis"):
        return {"score": _extract_analysis_score(generation)}
    return {"boxed": _last_boxed(generation)}


def _extract_proof_solution(generation: str) -> str:
    """Everything from the final ``## Solution`` heading onward (searched only post-``</think>`` so a
    ``## Solution`` mentioned inside reasoning isn't mistaken for the answer), stripped of stop tokens."""
    tail = generation.rsplit(_THINK_CLOSE, 1)[-1]
    start = tail.rfind("## Solution")
    if start < 0:
        return ""
    return tail[start:].replace(END_OF_TEXT, "").replace(END_OF_TURN, "").strip()


def _extract_analysis_score(generation: str) -> Optional[int]:
    """Final boxed grade BUCKETED to the nearest valid 0/1/6/7 (ties -> lower). None if no numeric box.

    Robust to LaTeX/text wrappers the grader sometimes emits — ``\\text{7}``, ``$7$``, ``7/7`` — by
    pulling the first integer out of the boxed content before bucketing (avoids dropping the score,
    which would otherwise shrink n_verifications / leave a solution unscored)."""
    raw = _last_boxed(generation)
    if raw is None:
        return None
    try:
        value = float(raw)
    except ValueError:
        m = re.search(r"-?\d+", raw)   # fallback: first integer inside e.g. \text{7}, $7$, 7/7
        if not m:
            return None
        value = float(m.group(0))
    return min(_VALID_ANALYSIS_SCORES, key=lambda g: (abs(g - value), g))


# --- OpenAI-compatible client (points at the vLLM server) -----------------------------------------


def build_client(server_addr: str, api_key: Optional[str], read_timeout: float = 300.0) -> OpenAI:
    base_url = server_addr if server_addr.startswith("http") else f"http://{server_addr}"
    if not base_url.endswith("/v1"):
        base_url += "/v1"
    key = api_key or os.environ.get("OPENAI_API_KEY", "sk-local")
    # A FINITE read timeout is essential: with streaming, `read` is the max SILENCE BETWEEN TOKENS (it
    # resets each chunk), independent of the per-rollout `deadline`. It is the stall safety net the
    # streaming-abort design relies on, so a non-positive value falls back to the safe default.
    read = read_timeout if (read_timeout and read_timeout > 0) else 300.0
    timeout = httpx.Timeout(connect=10.0, read=read, write=60.0, pool=read)
    # max_retries=0: the OpenAI SDK otherwise retries failed requests (default 2) with its OWN backoff
    # sleeps BEFORE create() returns — during which the rollout deadline cannot fire. On an unreachable
    # server that spins indefinitely (observed). With 0 retries, create() raises immediately, the rollout
    # records finish_reason="error" and the deadline stays in control.
    return OpenAI(base_url=base_url, api_key=key, timeout=timeout, max_retries=0)


# ==================================================================================================
# LAYER 3 — SOLVE PIPELINE (solution -> self-verify -> optional refine)
# ==================================================================================================

# Cap the candidate proof text fed into a judging/refining prompt (head-truncate) so the prompt stays
# within context. Applied to BOTH the self-verifier's candidate and the refiner's parent solution — a
# runaway/garbage solution (e.g. the raw-generation fallback can be 100k+ chars) must not overflow either
# prompt. Only the prompt copy is truncated; the full solution text is kept for best.txt / CSV / ranking.
CANDIDATE_CHAR_CAP = 80_000
VERIFY_SOLUTION_CHAR_CAP = CANDIDATE_CHAR_CAP  # back-compat alias
# Slack past a phase deadline before the pool-level backstop gives up (covers abort + sandbox teardown).
PHASE_BACKSTOP_SLACK_S = 60.0

DATAFRAME_COLUMNS = [
    "solution_id", "source", "session_index", "finish_reason", "truncated",
    "num_tool_calls", "solution_chars", "n_verifications", "scores",
    "mean_score", "median_score", "fully_verified", "is_refined", "has_text", "solution_text",
]


@dataclass
class SolveConfig:
    """Configuration for :func:`solve`. Defaults = the validated 8-solution champ."""

    # --- server / model ---
    server_addr: str
    api_key: Optional[str] = None
    model_name: str = DEFAULT_MODEL

    # --- output ---
    output_dir: str = "solve_output"

    # --- solution phase (default: tools only; set n_notool_solutions>0 to also use the no-tool prompt) ---
    n_tool_solutions: int = 8        # N1
    n_notool_solutions: int = 0      # N2
    solution_timeout_s: float = 1800.0   # T1

    # --- self-verification phase (model judges its own solutions, 0/1/6/7, reference-free) ---
    n_tool_verify: int = 3           # K1 (champ: 3; the analysis grader is a thinking pass needing ~250s,
    n_notool_verify: int = 0         # K2  so the cap is 360s, not lower)
    verify_timeout_s: float = 360.0      # T2

    # --- refinement phase (DEFAULT = the validated best setting: 1 round; refine the TOP-4 eligible
    #     solutions (eligible = neither unanimous-7 nor unanimous-0), 2 refine rollouts each; early-exit
    #     if any solution is already unanimous-7) ---
    refine_rounds: int = 1           # 0 disables refinement
    n_refine_candidates: int = 4     # refine the top-4 eligible solutions (by self-verify) — validated
    n_refine_tool: int = 2           # refine_tool rollouts per eligible candidate per round
    n_refine_notool: int = 0
    refine_timeout_s: float = 1800.0  # finite default; the CSV runner overrides this via the dynamic budget

    # Phase budgets (solution/verify/refine) accept "<=0 => no limit". The HTTP read timeout below is
    # NOT disable-able: always finite (the streaming-abort stall safety net).

    # --- execution ---
    max_parallel: int = 16
    sandbox_host: str = "127.0.0.1"
    sandbox_port: int = 6001
    stream_read_timeout_s: float = 300.0  # max silence between streamed tokens before a stall abort

    # --- sampling / engine ---
    temperature: float = 1.0
    top_p: float = 0.95
    max_tokens: int = 0              # <=0 => fill context window (intended)
    max_turns: int = 64
    python_timeout: int = 60         # per-call sandbox timeout (seconds)
    max_output_chars: int = 600      # tool-output cap — small: this model's context is tight


def _run_rollout(
    messages: list[dict], tools: Optional[list[dict]], cfg: SolveConfig, client,
    deadline: Optional[float] = None,
) -> dict[str, Any]:
    """Drive ONE rollout. Prompt-agnostic (callers build messages + extract afterwards). A sandbox is
    always provided so the model may execute a tool call regardless of whether the schema was shown."""
    sandbox = LocalSandbox(host=cfg.sandbox_host, port=cfg.sandbox_port)
    return asyncio.run(
        collect_rollout(
            client=client,
            model=cfg.model_name,
            messages=messages,
            tools=tools,
            sandbox=sandbox,
            temperature=cfg.temperature,
            top_p=cfg.top_p,
            max_tokens=cfg.max_tokens,
            max_turns=cfg.max_turns,
            python_timeout=cfg.python_timeout,
            max_output_chars=cfg.max_output_chars,
            deadline=deadline,
        )
    )


def _dump_trace(out_dir: Path, trace: dict[str, Any]) -> None:
    """Write a single trace to ``<out_dir>/<id>.json`` (best-effort)."""
    try:
        path = out_dir / f"{trace['id']}.json"
        with open(path, "w") as handle:
            json.dump(trace, handle, ensure_ascii=False, indent=2)
    except Exception as exc:
        log.warning("failed to dump trace %s: %s", trace.get("id"), exc)


def _run_phase(
    label: str,
    jobs: list[dict[str, Any]],
    runner: Callable[[dict[str, Any], Optional[float]], dict[str, Any]],
    out_dir: Path,
    deadline: Optional[float],
    cfg: SolveConfig,
) -> list[dict[str, Any]]:
    """Run ``jobs`` concurrently, dumping each as it finishes, bounded by the ABSOLUTE ``deadline``
    (epoch seconds; None = no limit). The same deadline is passed to each ``runner(job, deadline)`` so
    rollouts abort themselves at the cutoff; the pool wait adds PHASE_BACKSTOP_SLACK_S purely as a
    backstop. ``deadline`` already in the past => return [] without launching anything."""
    out_dir.mkdir(parents=True, exist_ok=True)
    if not jobs:
        return []
    if deadline is not None and time.time() >= deadline:
        log.info("[%s] deadline already passed — skipping phase", label)
        return []

    if deadline is not None:
        backstop_s: Optional[float] = max(0.0, deadline - time.time()) + PHASE_BACKSTOP_SLACK_S
    else:
        backstop_s = None
    completed: list[dict[str, Any]] = []
    seen: set = set()
    pool = ThreadPoolExecutor(max_workers=cfg.max_parallel)
    future_to_job = {pool.submit(runner, job, deadline): job for job in jobs}

    def _harvest(future) -> None:
        seen.add(future)
        job = future_to_job[future]
        try:
            trace = future.result()
        except Exception as exc:
            log.warning("[%s] job %s failed: %s: %s", label, job["id"], type(exc).__name__, exc)
            return
        _dump_trace(out_dir, trace)
        completed.append(trace)
        log.info("[%s] %d/%d done: %s", label, len(completed), len(jobs), trace["id"])

    try:
        for future in as_completed(future_to_job, timeout=backstop_s):
            _harvest(future)
    except FutureTimeoutError:
        salvaged = [f for f in future_to_job if f not in seen and f.done()]
        for f in salvaged:
            _harvest(f)
        log.warning("[%s] backstop fired at %.0fs: %d/%d returned (%d salvaged), abandoning the rest",
                    label, backstop_s, len(completed), len(jobs), len(salvaged))
    finally:
        pool.shutdown(wait=False, cancel_futures=True)
    return completed


def _solution_jobs(cfg: SolveConfig) -> list[dict[str, Any]]:
    jobs: list[dict[str, Any]] = []
    for i in range(cfg.n_tool_solutions):
        jobs.append({"id": f"sol_tool_{i}", "task": "proof_tool", "source": "tool", "session_index": i})
    for i in range(cfg.n_notool_solutions):
        jobs.append({"id": f"sol_notool_{i}", "task": "proof_notool", "source": "notool", "session_index": i})
    return jobs


def _generate_solutions(problem: str, cfg: SolveConfig, client, out_dir: Path,
                        deadline: Optional[float]) -> list[dict[str, Any]]:
    def runner(job: dict[str, Any], dl: Optional[float]) -> dict[str, Any]:
        messages, tools = build_messages(job["task"], problem)
        trace = _run_rollout(messages, tools, cfg, client, deadline=dl)
        trace.update(extract_result(job["task"], trace.get("generation", "")))
        trace["id"] = job["id"]
        trace["source"] = job["source"]
        trace["session_index"] = job["session_index"]
        trace["kind"] = "solution"
        return trace

    return _run_phase("solutions", _solution_jobs(cfg), runner, out_dir, deadline, cfg)


def _self_verify_solutions(
    problem: str, solutions: list[dict[str, Any]], cfg: SolveConfig, client, out_dir: Path,
    deadline: Optional[float],
) -> list[dict[str, Any]]:
    """The model scores each of its own solutions (0/1/6/7), reference-free. Correlates ~0.88 with
    oracle grades, so it is a reliable SELECTOR. No ground truth used."""
    solution_text_by_id = {s["id"]: s["solution_text"] for s in solutions}

    jobs: list[dict[str, Any]] = []
    for sol in solutions:
        for j in range(cfg.n_tool_verify):
            jobs.append({"id": f"{sol['id']}__verify_tool_{j}", "task": "analysis_tool",
                         "solution_id": sol["id"], "verifier": "tool"})
        for j in range(cfg.n_notool_verify):
            jobs.append({"id": f"{sol['id']}__verify_notool_{j}", "task": "analysis_notool",
                         "solution_id": sol["id"], "verifier": "notool"})

    def runner(job: dict[str, Any], dl: Optional[float]) -> dict[str, Any]:
        candidate = solution_text_by_id[job["solution_id"]][:CANDIDATE_CHAR_CAP]
        messages, tools = build_messages(job["task"], problem, candidate)
        trace = _run_rollout(messages, tools, cfg, client, deadline=dl)
        trace.update(extract_result(job["task"], trace.get("generation", "")))
        trace["id"] = job["id"]
        trace["solution_id"] = job["solution_id"]
        trace["verifier"] = job["verifier"]
        trace["kind"] = "self_verify"
        return trace

    return _run_phase("self_verify", jobs, runner, out_dir, deadline, cfg)


def _critique_text(trace: dict[str, Any]) -> str:
    """The written critique from a verification trace (post-</think>; the part with the fix info)."""
    g = trace.get("generation", "") or ""
    return (g.split("</think>")[-1] if "</think>" in g else g).strip()


def _grade_critiques(verify_traces: list[dict[str, Any]]) -> dict[str, list[tuple[int, str]]]:
    """solution_id -> list of (grade, critique_text) over its verification rollouts."""
    out: dict[str, list[tuple[int, str]]] = defaultdict(list)
    for t in verify_traces:
        score = t.get("score")
        if score is not None:
            out[t["solution_id"]].append((score, _critique_text(t)))
    return out


def _refine_eligible(scores: list[int]) -> bool:
    """Refine iff there is verification signal that is neither unanimous-7 nor unanimous-0."""
    return bool(scores) and not all(s == 7 for s in scores) and not all(s == 0 for s in scores)


def _any_fully_verified(solutions: list[dict[str, Any]],
                        grade_critiques: dict[str, list[tuple[int, str]]]) -> bool:
    """True if any non-empty solution is UNANIMOUSLY self-verified 7 (a confident answer exists)."""
    for s in solutions:
        if not s["solution_text"]:
            continue
        scores = [g for g, _ in grade_critiques.get(s["id"], [])]
        if scores and all(g == 7 for g in scores):
            return True
    return False


def _choose_critique(grade_critiques: list[tuple[int, str]]) -> str:
    """Pick the median-grade critique among ACTIONABLE grades (1 and 6 only; lower median on ties,
    longest at that grade). BOTH extremes are excluded: 0 = dismissive ("wrong, no progress") and
    7 = "correct, no changes needed" — neither carries fixable feedback. Returns "" if the candidate
    has no grade-1/6 critique (nothing actionable to refine on)."""
    actionable = [(g, t) for g, t in grade_critiques if 0 < g < 7 and t.strip()]
    if not actionable:
        return ""
    grades = sorted(g for g, _ in actionable)
    median_grade = grades[(len(grades) - 1) // 2]
    at_median = [t for g, t in actionable if g == median_grade]
    return max(at_median, key=len)


def _refine_solutions(
    problem: str, eligible: list[dict[str, Any]], grade_critiques: dict[str, list[tuple[int, str]]],
    cfg: SolveConfig, client, out_dir: Path, round_idx: int, deadline: Optional[float],
) -> list[dict[str, Any]]:
    """Refine each eligible solution, feeding its median-grade (non-zero) self-verification critique."""
    critique_by_parent: dict[str, str] = {}
    jobs: list[dict[str, Any]] = []
    for sol in eligible:
        critique = _choose_critique(grade_critiques.get(sol["id"], []))
        if not critique:
            continue
        critique_by_parent[sol["id"]] = critique
        for task, count, tag in (("refine_tool", cfg.n_refine_tool, "tool"),
                                 ("refine_notool", cfg.n_refine_notool, "notool")):
            for j in range(count):
                jobs.append({"id": f"{sol['id']}__refine{round_idx}_{tag}_{j}", "parent": sol["id"],
                             "parent_text": sol["solution_text"], "source": sol["source"],
                             "task": task, "tag": tag})

    def runner(job: dict[str, Any], dl: Optional[float]) -> dict[str, Any]:
        # Cap the parent proof (same as the verifier) so a runaway/garbage candidate can't overflow the
        # refine prompt's context. Full text is preserved elsewhere; only this prompt copy is truncated.
        parent_text = job["parent_text"][:CANDIDATE_CHAR_CAP]
        messages, tools = build_messages(
            job["task"], problem, parent_text, critique_by_parent[job["parent"]]
        )
        trace = _run_rollout(messages, tools, cfg, client, deadline=dl)
        trace.update(extract_result(job["task"], trace.get("generation", "")))
        trace["id"] = job["id"]
        trace["source"] = f"{job['source']}+refine_{job['tag']}"
        trace["session_index"] = 0
        trace["parent"] = job["parent"]
        trace["kind"] = "refine"
        return trace

    traces = _run_phase(f"refine_r{round_idx}", jobs, runner, out_dir, deadline, cfg)
    return [
        {"id": t["id"], "source": t["source"], "session_index": 0,
         "finish_reason": t.get("finish_reason"), "truncated": t.get("truncated"),
         "num_tool_calls": t.get("num_tool_calls", 0),
         "solution_text": (t.get("solution") or "").strip()}
        for t in traces
        if (t.get("solution") or "").strip()
    ]


def _build_dataframe(
    solutions: list[dict[str, Any]], verify_traces: list[dict[str, Any]]
) -> pd.DataFrame:
    """Aggregate self-verification scores per solution and return a best-first DataFrame."""
    scores_by_solution: dict[str, list[int]] = {s["id"]: [] for s in solutions}
    for trace in verify_traces:
        score = trace.get("score")
        if score is not None and trace["solution_id"] in scores_by_solution:
            scores_by_solution[trace["solution_id"]].append(score)

    rows = []
    for sol in solutions:
        scores = scores_by_solution[sol["id"]]
        text = sol["solution_text"]
        if not text:
            mean_score = median_score = 0.0   # empty solution is worth 0 by definition
        elif scores:
            mean_score = sum(scores) / len(scores)
            median_score = statistics.median(scores)
        else:
            mean_score = median_score = None  # verification budget elapsed before any score returned
        rows.append({
            "solution_id": sol["id"],
            "source": sol["source"],
            "session_index": sol["session_index"],
            "finish_reason": sol.get("finish_reason"),
            "truncated": sol.get("truncated"),
            "num_tool_calls": sol.get("num_tool_calls", 0),
            "solution_chars": len(text),
            "n_verifications": len(scores),
            "scores": scores,
            "mean_score": mean_score,
            "median_score": median_score,
            "fully_verified": bool(scores) and all(s == 7 for s in scores),
            "is_refined": "refine" in str(sol.get("source", "")),
            "has_text": bool(text),
            "solution_text": text,
        })

    frame = pd.DataFrame(rows, columns=DATAFRAME_COLUMNS)
    if frame.empty:
        return frame
    # Ranking, in order:
    #  1. has_text DESC — a NON-EMPTY solution ALWAYS outranks an empty one. Critical: never submit ""
    #     when any real (even unscored / 0-scored / truncated) solution exists. (Empty candidates get
    #     mean_score=0.0, which would otherwise beat a real-but-unverified candidate's None.)
    #  2. mean_score DESC — higher self-verify (None/unverified sorts last, but still above empties).
    #  3. is_refined ASC — on a self-verify tie, prefer the ORIGINAL over a refined candidate (an
    #     over-confident refined proof must not displace an equally-scored original).
    #  4/5. n_verifications, solution_chars DESC — more evidence, then longer (proxy for completeness).
    return frame.sort_values(
        by=["has_text", "mean_score", "is_refined", "n_verifications", "solution_chars"],
        ascending=[False, False, True, False, False],
        na_position="last",
    ).reset_index(drop=True)


def solve(problem_text: str, cfg: SolveConfig, problem_deadline: Optional[float] = None) -> dict[str, Any]:
    """Solve ``problem_text``, self-verify the solutions, and return the best one.

    ``problem_deadline`` (absolute epoch seconds, optional) is a SINGLE hard cutoff shared by ALL phases
    (solution + self-verify + refine + re-verify), matching the AIMO3 reference's one-deadline-per-problem
    contract. Each phase runs until its own sub-cap OR ``problem_deadline``, whichever comes first; a phase
    whose start is already past the deadline is skipped. So the whole problem is bounded by one budget and
    verify SHARES it. On timeout we still return the best solution generated so far (even unverified/
    truncated). ``None`` => only the per-phase sub-caps apply (no shared ceiling).

    Returns ``{"best": <str, "" only if NO non-empty solution was produced>, "all": <DataFrame>}``.
    """
    if isinstance(cfg, dict):
        cfg = SolveConfig(**cfg)

    client = build_client(cfg.server_addr, cfg.api_key, read_timeout=cfg.stream_read_timeout_s)
    output_dir = Path(cfg.output_dir)
    solutions_dir = output_dir / "solutions"
    verify_dir = output_dir / "self_verify"
    refine_dir = output_dir / "refine"

    def _deadline(cap: float) -> Optional[float]:
        """Absolute deadline for a phase with sub-cap ``cap`` (s), clamped to the hard problem_deadline."""
        d = (time.time() + cap) if (cap and cap > 0) else None
        if problem_deadline is not None:
            d = problem_deadline if d is None else min(d, problem_deadline)
        return d

    budget_str = (f"{problem_deadline - time.time():.0f}s left" if problem_deadline else "no shared deadline")
    log.info("solve: %d tool + %d notool solutions; self-verify %d/%d each; refine_rounds=%d; budget=%s "
             "(sub-caps sol=%.0fs verify=%.0fs refine=%.0fs) -> %s",
             cfg.n_tool_solutions, cfg.n_notool_solutions, cfg.n_tool_verify, cfg.n_notool_verify,
             cfg.refine_rounds, budget_str, cfg.solution_timeout_s, cfg.verify_timeout_s,
             cfg.refine_timeout_s, output_dir)

    def as_solution(t: dict[str, Any]) -> dict[str, Any]:
        return {"id": t["id"], "source": t["source"], "session_index": t["session_index"],
                "finish_reason": t.get("finish_reason"), "truncated": t.get("truncated"),
                "num_tool_calls": t.get("num_tool_calls", 0),
                "solution_text": (t.get("solution") or "").strip(),
                "generation": (t.get("generation") or "")}  # kept for the raw-generation fallback

    def self_verify(sols: list[dict[str, Any]]) -> list[dict[str, Any]]:
        to_verify = [s for s in sols if s["solution_text"]]
        if not to_verify:
            return []
        return _self_verify_solutions(problem_text, to_verify, cfg, client, verify_dir,
                                      _deadline(cfg.verify_timeout_s))

    # 1. Solve
    solutions = [as_solution(t) for t in
                 _generate_solutions(problem_text, cfg, client, solutions_dir, _deadline(cfg.solution_timeout_s))]
    log.info("solution phase: %d rollouts completed", len(solutions))
    if not solutions:
        log.warning("no solution rollout completed within the budget")
        return {"best": "", "all": pd.DataFrame(columns=DATAFRAME_COLUMNS)}

    # 2. Self-verify (shares the problem deadline; may be skipped/partial if the budget is spent)
    verify_traces = self_verify(solutions)
    log.info("self-verify phase: %d verification rollouts completed", len(verify_traces))

    # 3. Refine (disabled by default; refine_rounds=0)
    for rnd in range(cfg.refine_rounds):
        grade_critiques = _grade_critiques(verify_traces)

        # Early exit: if any solution is already UNANIMOUSLY self-verified 7, we have a confident answer
        # — return without refining (refine can only cost time and risk displacing it). Checked each
        # round, so a refined solution reaching all-7 also stops further rounds.
        if _any_fully_verified(solutions, grade_critiques):
            log.info("refine round %d: a solution is already unanimously self-verified 7 — "
                     "skipping refine", rnd)
            break

        def _sv_mean(sol: dict[str, Any]) -> float:
            grades = [sc for sc, _ in grade_critiques.get(sol["id"], [])]
            return sum(grades) / len(grades) if grades else -1.0

        eligible = [
            s for s in solutions
            if s["solution_text"]
            and _refine_eligible([sc for sc, _ in grade_critiques.get(s["id"], [])])
            and _choose_critique(grade_critiques.get(s["id"], []))
        ]
        eligible = sorted(eligible, key=_sv_mean, reverse=True)[:cfg.n_refine_candidates]
        if not eligible:
            log.info("refine round %d: nothing eligible (all unanimous-7 or -0)", rnd)
            break
        refined = _refine_solutions(problem_text, eligible, grade_critiques, cfg, client, refine_dir, rnd,
                                    _deadline(cfg.refine_timeout_s))
        if not refined:
            log.info("refine round %d: no refined solution produced", rnd)
            break
        new_verify = self_verify(refined)
        solutions += refined
        verify_traces += new_verify
        log.info("refine round %d: %d eligible -> %d refined (+%d verifications)",
                 rnd, len(eligible), len(refined), len(new_verify))

    frame = _build_dataframe(solutions, verify_traces)
    best = frame.iloc[0]["solution_text"] if not frame.empty else ""
    if not best:
        # Last resort: NO candidate reached "## Solution" (all truncated mid-reasoning / failed). Rather
        # than submit nothing, return our most-developed raw attempt so a partial proof can still earn
        # credit. Only original solution rollouts carry the raw generation; refined dicts don't (their
        # generation lives in their on-disk trace).
        raws = [(s.get("generation", "") or "").replace(END_OF_TEXT, "").replace(END_OF_TURN, "").strip()
                for s in solutions]
        raws = [r for r in raws if r]
        # Prefer the longest POST-</think> answer text (a real partial proof) over raw chain-of-thought;
        # only if no rollout produced any post-think text do we submit the longest think-block verbatim.
        post_think = [r.rsplit(_THINK_CLOSE, 1)[-1].strip() for r in raws]
        post_think = [p for p in post_think if p]
        if post_think:
            best = max(post_think, key=len)
            kind = "post-</think> partial answer"
        elif raws:
            best = max(raws, key=len)
            kind = "raw reasoning (no post-</think> text existed)"
        else:
            kind = None
        if best:
            log.warning("no solution reached '## Solution'; falling back to longest %s "
                        "(%d chars) so we still submit a partial attempt", kind, len(best))
    return {"best": best, "all": frame}


# ==================================================================================================
# CSV BATCH RUNNER — call solve() with the best config on each problem, one by one (Kaggle format)
# ==================================================================================================

# Accepted column names (first match wins), so the same runner works on iteration_set.csv,
# proofbench_v2.csv, etc.
_ID_COLUMNS = ("id", "Problem ID", "problem_id", "ID")
_PROBLEM_COLUMNS = ("problem", "Problem", "question", "Question", "statement")

# Wall-clock margin the per-problem watchdog allows on top of the budget/sub-caps before abandoning a
# (pathologically hung) solve() so the batch can still emit a row and continue.
WATCHDOG_SLACK_S = 300.0
_RESULTS_FIELDS = ["id", "solution", "best_self_verify", "best_chars", "elapsed_s", "error"]


def _pick(row: dict, names: tuple) -> Optional[str]:
    for n in names:
        if row.get(n):
            return row[n]
    return None


def _atomic_write_text(path: Path, text: str) -> None:
    """Write utf-8 text via a temp file + os.replace so a kill mid-write can't truncate the target."""
    tmp = path.with_suffix(path.suffix + ".tmp")
    with open(tmp, "w", encoding="utf-8", newline="") as f:
        f.write(text)
        f.flush()
        os.fsync(f.fileno())
    os.replace(tmp, path)


def _write_results_csv(path: Path, rows: list[dict]) -> None:
    """Render the results rows to CSV text (csv.DictWriter handles quoting) and write atomically."""
    import io
    buf = io.StringIO()
    w = csv.DictWriter(buf, fieldnames=_RESULTS_FIELDS)
    w.writeheader()
    w.writerows(rows)
    _atomic_write_text(path, buf.getvalue())


def _parse_endpoints_minimal(text: str) -> list[dict]:
    """Tiny fallback parser for the ENDPOINTS.yaml format (a list of ``- key: value`` blocks) so this
    file needs no PyYAML on Kaggle. Skips blank/``#`` lines."""
    out: list[dict] = []
    cur: Optional[dict] = None
    for line in text.splitlines():
        s = line.strip()
        if not s or s.startswith("#"):
            continue
        if s.startswith("- "):
            cur = {}
            out.append(cur)
            s = s[2:].strip()
        if cur is not None and ":" in s:
            k, _, v = s.partition(":")
            cur[k.strip()] = v.strip()
    return out


def load_active_endpoint(path: str = "ENDPOINTS.yaml") -> dict:
    """Return the single ACTIVE (uncommented) endpoint dict from ``path`` — keys: friendly_name, model,
    api_key, addr. Errors if there is not exactly one active entry (commented entries are ignored)."""
    text = Path(path).read_text()
    entries = None
    try:
        import yaml
        entries = yaml.safe_load(text)
    except Exception:
        entries = _parse_endpoints_minimal(text)
    entries = [e for e in (entries or []) if isinstance(e, dict) and e.get("addr")]
    if len(entries) != 1:
        raise ValueError(
            f"expected exactly ONE active endpoint in {path}, found {len(entries)} "
            f"(uncomment exactly one entry)"
        )
    return entries[0]


def main(argv: Optional[list[str]] = None) -> int:
    """Read a CSV of problems and run solve() on each, one at a time, with the best (default) config.

    The CSV needs an id column (one of: id / Problem ID / problem_id / ID) and a problem column
    (problem / Problem / question / statement). Per problem we write <outdir>/<id>/best.txt (the
    submission string) and all.json (ranked candidates), plus a running <outdir>/summary.jsonl.
    """
    ap = argparse.ArgumentParser(description="Run the best-config solver over a CSV of problems.")
    ap.add_argument("--csv", required=True, help="CSV with id + problem columns")
    ap.add_argument("--outdir", required=True)
    ap.add_argument("--out_csv", default=None,
                    help="results CSV path (id,solution,best_self_verify,best_chars,elapsed_s,error); "
                         "default <outdir>/results.csv. Written incrementally, one row per problem.")
    ap.add_argument("--sandbox_addr", required=True,
                    help="code-exec sandbox server: host:port (the stateful_python_code_exec endpoint)")
    # vLLM endpoint: by default pulled from the single active entry in --endpoints (ENDPOINTS.yaml).
    # Any of these flags overrides the corresponding YAML field.
    ap.add_argument("--endpoints", default="ENDPOINTS.yaml",
                    help="YAML with the active vLLM endpoint (friendly_name/model/api_key/addr)")
    ap.add_argument("--server_addr", default=None, help="vLLM host:port/URL (overrides ENDPOINTS.yaml addr)")
    ap.add_argument("--api_key", default=None, help="overrides ENDPOINTS.yaml api_key")
    ap.add_argument("--model_name", default=None, help="overrides ENDPOINTS.yaml model")
    ap.add_argument("--max_parallel", type=int, default=16)
    # Overrides (default = the validated champ config baked into SolveConfig).
    ap.add_argument("--n_tool_solutions", type=int, default=SolveConfig.n_tool_solutions)
    ap.add_argument("--n_tool_verify", type=int, default=SolveConfig.n_tool_verify)
    ap.add_argument("--refine_rounds", type=int, default=SolveConfig.refine_rounds)
    ap.add_argument("--n_refine_candidates", type=int, default=SolveConfig.n_refine_candidates)
    ap.add_argument("--n_refine_tool", type=int, default=SolveConfig.n_refine_tool)
    ap.add_argument("--solution_timeout_s", type=float, default=SolveConfig.solution_timeout_s,
                    help="fixed per-problem solution cap (used when --total_budget_s <= 0)")
    ap.add_argument("--verify_timeout_s", type=float, default=SolveConfig.verify_timeout_s)
    ap.add_argument("--refine_timeout_s", type=float, default=SolveConfig.refine_timeout_s,
                    help="fixed per-problem refine cap (used when --total_budget_s <= 0)")
    # Dynamic time budget: with --total_budget_s > 0, each problem's solution cap is computed at runtime
    # as clamp(remaining_total / remaining_problems, min, max) - verify_reserve. Fast problems underspend,
    # so their leftover rolls forward to later problems automatically (e.g. if problem 1 finishes in
    # 10 min, the other 5 each get (budget - 10 min) / 5).
    ap.add_argument("--total_budget_s", type=float, default=0.0,
                    help="total wall-clock budget across ALL problems; >0 enables dynamic per-problem caps")
    ap.add_argument("--verify_reserve_s", type=float, default=720.0,
                    help="time reserved per problem for verify+overhead (subtracted from the allocation)")
    ap.add_argument("--min_problem_s", type=float, default=1800.0,
                    help="floor on a problem's total allocation in dynamic mode")
    ap.add_argument("--max_problem_s", type=float, default=5400.0,
                    help="ceiling on a problem's total allocation in dynamic mode")
    args = ap.parse_args(argv)

    logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s")
    logging.getLogger("httpx").setLevel(logging.WARNING)

    # Resolve the vLLM endpoint: CLI flags win; otherwise fall back to the active ENDPOINTS.yaml entry.
    server_addr, api_key, model_name = args.server_addr, args.api_key, args.model_name
    if server_addr is None or api_key is None or model_name is None:
        ep = load_active_endpoint(args.endpoints)
        server_addr = server_addr or ep.get("addr")
        api_key = api_key or ep.get("api_key")
        model_name = model_name or ep.get("model") or DEFAULT_MODEL
        log.info("endpoint %s: model=%s addr=%s", ep.get("friendly_name"), model_name, server_addr)
    if not server_addr:
        ap.error("no vLLM server_addr (pass --server_addr or set an active entry in ENDPOINTS.yaml)")

    # Split the sandbox "host:port" (port optional -> default 6001).
    sb = args.sandbox_addr.split("://")[-1]  # tolerate an accidental scheme prefix
    sandbox_host, _, sandbox_port_s = sb.partition(":")
    sandbox_port = int(sandbox_port_s) if sandbox_port_s else 6001

    out = Path(args.outdir)
    out.mkdir(parents=True, exist_ok=True)
    with open(args.csv) as handle:
        rows = list(csv.DictReader(handle))

    # Pre-filter to valid (id, problem) pairs so the dynamic remaining-count is correct. Invalid input
    # rows are NOT silently dropped — they get an error row in the output CSV (csv in -> csv out: every
    # input line must appear in the output). Duplicate ids are disambiguated with a suffix so their
    # per-problem dirs / best.txt don't collide and both rows survive.
    problems: list[tuple[str, str]] = []
    invalid_rows: list[dict] = []
    seen_ids: dict[str, int] = {}
    for idx, row in enumerate(rows):
        pid = _pick(row, _ID_COLUMNS)
        problem = _pick(row, _PROBLEM_COLUMNS)
        if not pid or not (problem and problem.strip()):
            synth_id = str(pid) if pid else f"_row{idx}"
            log.warning("input row %d has missing id/problem -> emitting error row id=%s: %.80s",
                        idx, synth_id, json.dumps(row, ensure_ascii=False))
            invalid_rows.append({"id": synth_id, "solution": "", "best_self_verify": "",
                                 "best_chars": 0, "elapsed_s": 0, "error": "missing id or problem text"})
            continue
        pid = str(pid)
        if pid in seen_ids:
            seen_ids[pid] += 1
            new_pid = f"{pid}__dup{seen_ids[pid]}"
            log.warning("duplicate id %s in %s -> disambiguating to %s so outputs don't collide",
                        pid, args.csv, new_pid)
            pid = new_pid
        else:
            seen_ids[pid] = 0
        problems.append((pid, problem))

    if not problems:
        log.error("no usable problems in %s (need an id column %s and a problem column %s)",
                  args.csv, _ID_COLUMNS, _PROBLEM_COLUMNS)
        # Still emit any error rows so the output CSV reflects every input line.
        if invalid_rows:
            out_csv0 = Path(args.out_csv) if args.out_csv else (out / "results.csv")
            out_csv0.parent.mkdir(parents=True, exist_ok=True)
            _write_results_csv(out_csv0, invalid_rows)
        return 1

    if args.total_budget_s > 0:
        log.info("dynamic budget (greedy + reserved floor, AIMO3-harness style): %.0fs total over %d "
                 "problems; each problem takes time_left minus a %.0fs floor reserved per other remaining "
                 "problem, clamped to [%.0fs, %.0fs]; verify_reserve=%.0fs",
                 args.total_budget_s, len(problems), args.min_problem_s, args.min_problem_s,
                 args.max_problem_s, args.verify_reserve_s)

    summary: list[dict] = []
    results_rows: list[dict] = list(invalid_rows)  # invalid input lines carried through to the CSV
    out_csv = Path(args.out_csv) if args.out_csv else (out / "results.csv")
    out_csv.parent.mkdir(parents=True, exist_ok=True)
    t_start_all = time.time()
    for i, (pid, problem) in enumerate(problems):
        # Dynamic per-problem budget — GREEDY-with-reserved-floor, matching the Kaggle harness
        # (AIMO3 44-50-let-me-over-cook): this problem may use ALL remaining time EXCEPT a reserved
        # floor (min_problem_s) for each OTHER remaining problem, capped at the ceiling (max_problem_s)
        # and floored at min_problem_s. Front-loads time to early problems while guaranteeing every
        # later problem >= floor. Then carve the per-problem budget into the long poles: verify_reserve
        # covers the verify pass(es)+overhead; with refine on, solution AND refine each get HALF the rest.
        sol_timeout = args.solution_timeout_s
        refine_timeout = args.refine_timeout_s
        problem_deadline: Optional[float] = None
        budget = 0.0
        if args.total_budget_s > 0:
            time_left = args.total_budget_s - (time.time() - t_start_all)
            problems_left_others = max(0, (len(problems) - i) - 1)
            reserved = problems_left_others * args.min_problem_s
            budget = time_left - reserved
            budget = min(budget, args.max_problem_s)   # ceiling
            budget = max(budget, args.min_problem_s)   # floor
            # ONE hard deadline for the WHOLE problem (solution + verify + refine + re-verify all share
            # it) — this, not the per-phase sub-caps, is what actually bounds per-problem wall-clock.
            problem_deadline = time.time() + budget
            # Sub-caps carve the budget into the long poles (soft; clamped to problem_deadline in solve()).
            gen_budget = max(120.0, budget - args.verify_reserve_s)
            if args.refine_rounds > 0:
                sol_timeout = refine_timeout = gen_budget / 2.0
            else:
                sol_timeout = gen_budget
            log.info("[budget] time_left=%.0fs reserve_others=%.0fs -> budget=%.0fs (deadline shared by "
                     "all phases) -> sub-caps solution=%.0fs refine=%.0fs verify=%.0fs",
                     time_left, reserved, budget, sol_timeout,
                     refine_timeout if args.refine_rounds > 0 else 0.0, args.verify_timeout_s)
        pdir = out / str(pid)
        cfg = SolveConfig(
            server_addr=server_addr, api_key=api_key, model_name=model_name,
            output_dir=str(pdir), max_parallel=args.max_parallel,
            sandbox_host=sandbox_host, sandbox_port=sandbox_port,
            n_tool_solutions=args.n_tool_solutions, n_tool_verify=args.n_tool_verify,
            refine_rounds=args.refine_rounds, n_refine_candidates=args.n_refine_candidates,
            n_refine_tool=args.n_refine_tool, solution_timeout_s=sol_timeout,
            verify_timeout_s=args.verify_timeout_s, refine_timeout_s=refine_timeout,
        )
        print(f"\n{'#' * 70}\n# {pid}  starting\n{'#' * 70}", flush=True)
        t0 = time.time()
        best_text = ""
        err = ""
        best_sv = None
        df = None
        # Hard wall-clock watchdog around solve(): even though solve() honors `problem_deadline`
        # cooperatively, run it in a worker thread and abandon it if it overruns by a wide margin, so a
        # pathological hang can never stop us from emitting a row for THIS problem and moving to the next.
        wd_timeout = ((problem_deadline - time.time()) if problem_deadline is not None
                      else (sol_timeout + refine_timeout + 2 * args.verify_timeout_s)) + WATCHDOG_SLACK_S
        # NOTE: do NOT use `with ThreadPoolExecutor(...)` — its __exit__ does shutdown(wait=True), which
        # would block forever on a truly hung solve(), defeating the watchdog. Shut down with wait=False
        # so we abandon the orphaned worker and move on (it cooperatively dies at its own deadline).
        solve_pool = ThreadPoolExecutor(max_workers=1)
        try:
            fut = solve_pool.submit(solve, problem, cfg, problem_deadline=problem_deadline)
            result = fut.result(timeout=max(60.0, wd_timeout))
            elapsed = time.time() - t0
            best_text = result.get("best") or ""
            df = result["all"]
            top = df.iloc[0] if len(df) else None
            if top is not None and top["mean_score"] is not None and top["mean_score"] == top["mean_score"]:
                best_sv = float(top["mean_score"])
        except FutureTimeoutError:
            elapsed = time.time() - t0
            err = f"watchdog: solve() exceeded {wd_timeout:.0f}s wall-clock (abandoned)"
            logging.error("[%s] %s", pid, err)
        except Exception as exc:  # one problem's failure must not abort the whole batch
            elapsed = time.time() - t0
            err = repr(exc)
            logging.exception("[%s] solve() failed after %.0fs", pid, elapsed)
        finally:
            solve_pool.shutdown(wait=False, cancel_futures=True)

        # ALWAYS write best.txt (even "" on failure/timeout) — overwrites any stale file from a prior run
        # so the per-problem artifact never lies, and guarantees an artifact exists for every problem.
        try:
            pdir.mkdir(parents=True, exist_ok=True)
            (pdir / "best.txt").write_text(best_text, encoding="utf-8")
            if df is not None:
                df.to_json(pdir / "all.json", orient="records", force_ascii=False)
        except Exception:
            logging.exception("[%s] could not write best.txt/all.json", pid)
        row_out = {"id": str(pid), "order": i, "solution_timeout_s": round(sol_timeout, 1),
                   "refine_timeout_s": round(refine_timeout, 1), "elapsed_s": round(elapsed, 1),
                   "n_candidates": int(len(df)) if df is not None else 0,
                   "best_self_verify": best_sv, "best_chars": len(best_text), "error": err}
        summary.append(row_out)
        results_rows.append({"id": str(pid), "solution": best_text,
                             "best_self_verify": "" if best_sv is None else best_sv,
                             "best_chars": len(best_text), "elapsed_s": round(elapsed, 1), "error": err})
        # CSV/summary flush, GUARDED so a transient write error can't abort the whole batch. Written
        # atomically (tmp + os.replace) each iteration so a kill mid-write can't truncate prior rows.
        try:
            _atomic_write_text(out / "summary.jsonl",
                               "".join(json.dumps(s, ensure_ascii=False) + "\n" for s in summary))
            _write_results_csv(out_csv, results_rows)
        except Exception:
            logging.exception("[%s] could not flush summary/results CSV", pid)
        print(f"[{pid}] elapsed={row_out['elapsed_s']:.0f}s cands={row_out['n_candidates']} "
              f"best_sv={row_out['best_self_verify']} best_chars={row_out['best_chars']}"
              f"{' ERR='+err if err else ''}", flush=True)

    try:
        (out / "DONE.txt").write_text(f"finished {len(summary)} problems\n", encoding="utf-8")
    except Exception:
        pass
    print(f"\n=== DONE ({len(summary)} problems) -> {out_csv} ===", flush=True)
    return 0


if __name__ == "__main__":
    raise SystemExit(main())


In [ ]:
TEST_CSV_PATH = "/kaggle/input/competitions/ai-mathematical-olympiad-proof-pilot/test.csv"
DEBUG_CSV_PATH = "/kaggle/input/datasets/chankhavu/aimo-proof-pilot-debug-problems/easier-6.csv"


test_df = pd.read_csv(TEST_CSV_PATH)
debug_df = pd.read_csv(DEBUG_CSV_PATH)
display(test_df)
display(debug_df)

# if len(test_df) == 3:
#     debug_df = debug_df.iloc[1:2]
#     display(debug_df)
#     debug_df[["id", "problem"]].to_csv("proofbench_test.csv", index=False)
#     TEST_CSV_PATH = "proofbench_test.csv"
print(TEST_CSV_PATH)

In [ ]:
!python kaggle_solver.py \
    --csv $TEST_CSV_PATH \
    --out_csv output_solutions.csv \
    --outdir /kaggle/working/artifacts \
    --sandbox_addr 127.0.0.1:6000 \
    --server_addr 127.0.0.1:8000 \
    --api_key "sk-local" \
    --model_name "chankhavu/smolmo-32b-sft-merged-proofpilot-nvfp4" \
    --total_budget_s 25200 \
    --min_problem_s 1800 \
    --max_problem_s 7000 \
    --verify_reserve_s 2000 \
    --verify_timeout_s 800 \
    --n_tool_solutions 16 \
    --n_tool_verify 3 \
    --n_refine_candidates 4 \
    --refine_rounds 1 \
    --n_refine_tool 4

In [ ]:
submission_df = pd.read_csv("output_solutions.csv")
display(submission_df)
print(submission_df["solution"].values[0])

In [ ]:
submission_df[["id", "solution"]].rename(columns={"solution": "answer"}).to_csv("submission.csv", index=False)

In [ ]:
!ls

In [ ]:
%%writefile check_sandbox.py
#!/usr/bin/env python3
"""Standalone sanity check for the code-execution sandbox used by kaggle_solver.py.

Exercises the SAME path the solver uses (sandbox_client.LocalSandbox.execute_code) and checks:
  1. basic execution + stdout capture
  2. stateful session (ipython keeps variables across calls)
  3. error/traceback is returned (not swallowed)
  4. the `timeout` argument is honored
  5. output is truncated to max_output_characters
Then cleans up the session and closes the client.

Usage:
    python3 check_sandbox.py                 # defaults to 127.0.0.1:6001
    python3 check_sandbox.py 127.0.0.1:6000  # or pass host:port
Exit code 0 = all checks passed, 1 = something failed.
"""
import asyncio
import sys
import uuid

from sandbox_client import LocalSandbox


def _stdout(out: dict) -> str:
    return (out.get("stdout") or "")


def _stderr(out: dict) -> str:
    return (out.get("stderr") or "")


async def main(host: str, port: str) -> int:
    print(f"[*] sandbox target: http://{host}:{port}")
    sb = LocalSandbox(host=host, port=port)
    sid = str(uuid.uuid4())
    passed, failed = 0, 0

    def check(name: str, ok: bool, detail: str = "") -> None:
        nonlocal passed, failed
        if ok:
            passed += 1
            print(f"  [PASS] {name}")
        else:
            failed += 1
            print(f"  [FAIL] {name}  {detail}")

    try:
        # 1. basic execution + stdout
        try:
            out, _ = await sb.execute_code(
                generated_code="print(6 * 7)", session_id=sid, timeout=15, max_output_characters=1000
            )
            check("1. basic execution / stdout", _stdout(out).strip() == "42",
                  f"got stdout={_stdout(out)!r} stderr={_stderr(out)!r}")
        except Exception as exc:
            check("1. basic execution / stdout", False, f"raised {type(exc).__name__}: {exc} "
                  "(server unreachable? wrong port? sandbox not running?)")
            print("\n[!] could not execute at all — aborting further checks.")
            return 1

        # 2. stateful session: define a var, then use it in a later call (ipython keeps state)
        await sb.execute_code(generated_code="x = 123", session_id=sid, timeout=15, max_output_characters=1000)
        out, _ = await sb.execute_code(
            generated_code="print(x + 1)", session_id=sid, timeout=15, max_output_characters=1000
        )
        check("2. stateful session (var persists)", _stdout(out).strip() == "124",
              f"got stdout={_stdout(out)!r} (state not preserved across calls?)")

        # 3. error/traceback surfaces
        out, _ = await sb.execute_code(
            generated_code="1/0", session_id=sid, timeout=15, max_output_characters=2000
        )
        combined = _stdout(out) + _stderr(out)
        check("3. error/traceback returned", "ZeroDivisionError" in combined,
              f"got {combined!r}")

        # 4. timeout is honored: sleep longer than the timeout -> should time out, not hang forever
        import time as _t
        t0 = _t.time()
        out, _ = await sb.execute_code(
            generated_code="import time; time.sleep(10)", session_id=sid, timeout=3, max_output_characters=1000
        )
        elapsed = _t.time() - t0
        timed_out = ("timeout" in str(out.get("process_status", "")).lower()
                     or "timed out" in _stderr(out).lower() or elapsed < 8)
        check("4. timeout honored (~3s cap on a 10s sleep)", timed_out,
              f"elapsed={elapsed:.1f}s status={out.get('process_status')!r} stderr={_stderr(out)!r}")

        # 5. output truncation to max_output_characters
        out, _ = await sb.execute_code(
            generated_code="print('A' * 5000)", session_id=sid, timeout=15, max_output_characters=200
        )
        check("5. output truncated to max_output_characters", len(_stdout(out)) <= 600,
              f"got {len(_stdout(out))} chars (cap was 200)")

    finally:
        try:
            await sb.delete_session(sid)
        except Exception as exc:
            print(f"  [warn] delete_session failed: {type(exc).__name__}: {exc}")
        try:
            await sb.close()
        except Exception:
            pass

    print(f"\n[*] {passed} passed, {failed} failed")
    if failed == 0:
        print("[OK] sandbox is working.")
        return 0
    print("[X] sandbox has problems — see failures above.")
    return 1


if __name__ == "__main__":
    addr = sys.argv[1] if len(sys.argv) > 1 else "127.0.0.1:6001"
    addr = addr.split("://")[-1]
    h, _, p = addr.partition(":")
    raise SystemExit(asyncio.run(main(h, p or "6001")))


In [ ]:
# !python check_sandbox.py 127.0.0.1:6000